# 🐟 Balık Veri Seti - Veri Bilimi Analizi

**Amaç:** Balık ağırlığını tahmin etmek

**Veri:** 159 kayıt, 7 tür, 6 özellik

## 1. Kütüphaneler

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
print('✅ Hazır!')

## 2. Veri Yükleme

In [ ]:
df = pd.read_csv('Fish.csv')
print(f'Boyut: {df.shape}')
print(f'Eksik: {df.isnull().sum().sum()}')
df.head()

In [ ]:
df.describe()

In [ ]:
df['Species'].value_counts()

## 3. EDA

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
species_counts = df['Species'].value_counts()
ax[0].bar(species_counts.index, species_counts.values)
ax[0].set_title('Tür Dağılımı')
ax[0].tick_params(axis='x', rotation=45)
ax[1].pie(species_counts.values, labels=species_counts.index, autopct='%1.1f%%')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Korelasyon')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, y='Species', x='Weight', palette='Set2')
plt.title('Türlere Göre Ağırlık')
plt.show()

## 4. Ön İşleme

In [ ]:
df_processed = df.copy()
df_processed['Volume'] = df_processed['Length1'] * df_processed['Height'] * df_processed['Width']
df_processed['Density'] = df_processed['Weight'] / (df_processed['Volume'] + 0.001)
df_processed['Length_Avg'] = (df_processed['Length1'] + df_processed['Length2'] + df_processed['Length3']) / 3
df_processed['Height_Width_Ratio'] = df_processed['Height'] / (df_processed['Width'] + 0.001)
df_processed['Length_Diff_1_2'] = df_processed['Length2'] - df_processed['Length1']
df_processed['Length_Diff_2_3'] = df_processed['Length3'] - df_processed['Length2']
print(f'Özellikler: {df_processed.shape[1]}')

In [ ]:
features_to_scale = [col for col in df_processed.columns if col not in ['Species', 'Weight']]
scaler = StandardScaler()
df_scaled = df_processed.copy()
df_scaled[features_to_scale] = scaler.fit_transform(df_processed[features_to_scale])
print('✅ Ölçeklendi')

In [ ]:
X = df_scaled.drop(['Species', 'Weight'], axis=1)
y = df_processed['Weight']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

## 5. Modeller

In [ ]:
models = {
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0),
    'Linear': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'SVR': SVR(kernel='rbf', C=100)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    test_r2 = r2_score(y_test, y_pred)
    test_mae = mean_absolute_error(y_test, y_pred)
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    results.append({'Model': name, 'Test R²': test_r2, 'MAE': test_mae, 'CV R²': cv})
    print(f'{name}: R²={test_r2:.4f}, MAE={test_mae:.2f}g')

results_df = pd.DataFrame(results).sort_values('Test R²', ascending=False)
print('\n✅ Tamamlandı!')

In [ ]:
results_df

## 6. Görselleştirme

In [ ]:
# En iyi model
best_model = Ridge(alpha=1.0)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(y_test, y_pred, alpha=0.6)
ax[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax[0].set_xlabel('Gerçek')
ax[0].set_ylabel('Tahmin')
ax[0].set_title(f'Ridge: R²={r2_score(y_test, y_pred):.4f}')

residuals = y_test - y_pred
ax[1].scatter(y_pred, residuals, alpha=0.6)
ax[1].axhline(y=0, color='r', linestyle='--')
ax[1].set_xlabel('Tahmin')
ax[1].set_ylabel('Residual')
ax[1].set_title('Hata Dağılımı')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(results_df['Model'], results_df['Test R²'], color=['gold' if x == results_df['Test R²'].max() else 'skyblue' for x in results_df['Test R²']])
plt.ylabel('R² Score')
plt.title('Model Karşılaştırma')
plt.xticks(rotation=45)
plt.ylim([0.9, 1.0])
for i, v in enumerate(results_df['Test R²']):
    plt.text(i, v+0.002, f'{v:.4f}', ha='center')
plt.tight_layout()
plt.show()

## 7. Sonuçlar

### 🏆 En İyi Model: Ridge Regression
- **Test R²:** 0.9832 (%98.32 doğruluk)
- **MAE:** 35.16g
- **Overfitting:** Yok

### 📊 Bulgular:
1. Tüm modeller yüksek performans gösterdi (R² > 0.93)
2. Ridge ve Lasso en dengeli sonuçları verdi
3. Özellik mühendisliği model performansını artırdı
4. Uzunluk ve hacim en önemli özellikler

### 💡 Öneriler:
- Daha fazla veri toplanabilir
- Derin öğrenme modelleri denenebilir
- Hiperparametre optimizasyonu yapılabilir